# AlphaTrain: Frozen Backbone Value Training (Colab)

Train **only the Value Head** while freezing the backbone + policy head.
Tests whether the existing backbone features can support value prediction
without destroying the policy (which regressed in iterations 1a-1c).

**Policy stays at ~314 baseline (frozen). Only value head learns.**

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code (84 KB, **rebuilt**)
2. `alphatrain_pairwise.pt` — expert training data (already on Drive)
3. `alphatrain_td_best.pt` — base model (already on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy data + model
os.makedirs('/content/alphatrain/data', exist_ok=True)
for f in ['alphatrain_pairwise.pt', 'alphatrain_td_best.pt']:
    src = os.path.join(DRIVE, f)
    dst = os.path.join('/content/alphatrain/data', f)
    if not os.path.exists(dst):
        print(f'Copying {f}...')
        shutil.copy(src, dst)
    print(f'{f}: {os.path.getsize(dst)/1e6:.0f} MB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')
    if g.total_memory < 20e9:
        print('WARNING: T4 may not have enough VRAM. Use A100 runtime.')

!cd /content && python -m pytest alphatrain/tests/test_model.py alphatrain/tests/test_observation.py -v --tb=short 2>&1 | tail -10

In [ ]:
# Frozen backbone: train value head only on expert pairwise data
# Backbone + policy head frozen — policy guaranteed to stay at ~314
# Value head learns pairwise ranking + anchor MSE
%cd /content
!python -m alphatrain.train \
    --tensor-file alphatrain/data/alphatrain_pairwise.pt \
    --gpu-data --amp --compile \
    --value-bins 1 --anchor-weight 0.001 --rank-weight 1.0 \
    --resume alphatrain/data/alphatrain_td_best.pt --warm-start \
    --freeze-backbone \
    --epochs 10 --batch-size 8192 --lr 1e-3 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/frozen_value_best.pt \
    --save-dir /content/checkpoints/frozen_value

In [ ]:
# Evaluate policy (should be ~314 unchanged since backbone is frozen)
%cd /content
!python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/frozen_value/best.pt \
    --games 50 --seed 42

In [ ]:
# Copy final model to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/frozen_value/{f}'
    dst = f'{DRIVE}/frozen_value_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst}')